In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

In [2]:
# ─── CONFIG ───────────────────────────────────────────────────────────────────
INPUT_PATH  = "D:/Lomba/DATATHON DICODING/inflasiwatch/data/final/df_merged_final.csv"
OUTPUT_DIR  = Path("D:/Lomba/DATATHON DICODING/inflasiwatch/data/final/output_clean")
OUTPUT_DIR.mkdir(exist_ok=True)
 
RANDOM_SEED = 42
TARGET_COL  = "ihk_umum_mtm"
 
SECONDARY_TARGETS = [
    "inflasi_general_mtm",
    "inflasi_core_mtm",
    "inflasi_administered_price_mtm",
    "inflasi_volatile_good_mtm",
]

In [3]:
# ─── STEP 1: LOAD DATA ────────────────────────────────────────────────────────
print("=" * 60)
print("STEP 1: LOAD DATA")
print("=" * 60)
 
df = pd.read_csv(INPUT_PATH)
print(f"Shape awal       : {df.shape}")
print(f"Kota             : {sorted(df['kota'].unique().tolist())}")
print(f"Date range       : {df['date'].min()} → {df['date'].max()}")
 
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["kota", "date"]).reset_index(drop=True)
 
report_lines = []
report_lines.append(f"Shape awal: {df.shape}")
report_lines.append(f"Total missing awal: {df.isnull().sum()}")

STEP 1: LOAD DATA
Shape awal       : (375, 181)
Kota             : ['bandung', 'jakarta', 'makassar', 'medan', 'surabaya']
Date range       : 2020/01/01 → 2026/03/01


In [4]:
# ─── STEP 2: DROP BARIS TARGET NaN ────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 2: DROP BARIS PERTAMA PER KOTA (TARGET NaN ARTIFACT)")
print("=" * 60)
 
# Baris pertama tiap kota NaN di ihk_umum_mtm karena lag → wajib drop, bukan impute
n_before = len(df)
target_nan_mask = df[TARGET_COL].isna()
print(f"Baris dengan target NaN: {target_nan_mask.sum()} (1 per kota — wajar)")
print(df[target_nan_mask][["date", "kota", TARGET_COL]])
 
df = df[~target_nan_mask].reset_index(drop=True)
print(f"\nShape setelah drop target NaN: {df.shape} (hilang {n_before - len(df)} baris)")
report_lines.append(f"Baris di-drop (target NaN): {n_before - len(df)}")


STEP 2: DROP BARIS PERTAMA PER KOTA (TARGET NaN ARTIFACT)
Baris dengan target NaN: 5 (1 per kota — wajar)
          date      kota  ihk_umum_mtm
0   2020-01-01   bandung           NaN
75  2020-01-01   jakarta           NaN
150 2020-01-01  makassar           NaN
225 2020-01-01     medan           NaN
300 2020-01-01  surabaya           NaN

Shape setelah drop target NaN: (370, 181) (hilang 5 baris)


In [5]:
# ─── STEP 3: HANDLE MISSING VALUE ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 3: HANDLE MISSING VALUE")
print("=" * 60)
 
miss_before = df.isnull().sum()
miss_cols   = miss_before[miss_before > 0].sort_values(ascending=False)
print(f"Kolom dengan missing sebelum impute: {len(miss_cols)}")
print(miss_cols.to_string())
 
# Kategorisasi kolom
lag_cols      = [c for c in df.columns if "_lag" in c]
roll_cols     = [c for c in df.columns if "_roll_" in c]
diff_pct_cols = [c for c in df.columns if "_diff1" in c or "_pct_change" in c]
macro_cols    = ["kurs_usd_idr", "bi_rate_percent", "harga_bbm_pertamax", "oceanic_nino_index"]
ihk_kel_cols  = [c for c in df.columns if c.startswith("ihk_kel_")]
flag_cols     = [c for c in df.columns if c.startswith("is_") or c.endswith("_event")]
 
# Kumpulkan semua kolom numerik yang perlu di-impute per kota
all_fill_cols = list(set(lag_cols + roll_cols + diff_pct_cols + macro_cols + ihk_kel_cols))
all_fill_cols = [c for c in all_fill_cols if c in df.columns]
 
# Impute per kota: forward fill → backward fill (kronologis, aman anti-leakage)
# groupby + transform adalah cara idiomatis pandas dan paling aman
print(f"\nMengimpute {len(all_fill_cols)} kolom dengan ffill → bfill per kota...")
df[all_fill_cols] = (
    df.groupby("kota")[all_fill_cols]
    .transform(lambda x: x.ffill().bfill())
)
 
# Flag/binary: NaN = 0 (tidak ada event)
for c in flag_cols:
    if c in df.columns:
        df[c] = df[c].fillna(0).astype(int)
 
miss_after = df.isnull().sum().sum()
print(f"Total missing setelah impute: {miss_after}")
if miss_after > 0:
    print("Kolom masih missing:")
    print(df.isnull().sum()[df.isnull().sum() > 0])
report_lines.append(f"Total missing setelah impute: {miss_after}")


STEP 3: HANDLE MISSING VALUE
Kolom dengan missing sebelum impute: 117
gt_harga_daging_ayam_lag3_pct_change          127
gt_harga_daging_ayam_lag2_pct_change          124
gt_harga_daging_ayam_lag1_pct_change          122
gt_harga_daging_ayam_pct_change               119
gt_harga_cabai_lag3_pct_change                 39
gt_harga_cabai_lag2_pct_change                 34
gt_harga_cabai_lag1_pct_change                 29
ihk_umum_mtm_roll_std_6                        25
ihk_umum_mtm_roll_mean_6                       25
gt_harga_cabai_pct_change                      24
gt_harga_minyak_goreng_lag3_pct_change         23
inflasi_general_mtm_roll_mean_6                20
inflasi_general_mtm_roll_std_6                 20
inflasi_core_mtm_roll_mean_6                   20
inflasi_core_mtm_roll_std_6                    20
inflasi_administered_price_mtm_roll_mean_6     20
inflasi_administered_price_mtm_roll_std_6      20
inflasi_volatile_good_mtm_roll_mean_6          20
inflasi_volatile_good_mtm_rol

In [6]:
# ─── STEP 4: CLIP OUTLIER EKSTREM ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 4: DETEKSI & CLIP OUTLIER EKSTREM")
print("=" * 60)
 
# Target utama TIDAK di-clip — nilai -10.82 adalah deflasi COVID nyata, model harus belajar ini
# Hanya fitur input yang di-clip dengan 1%-99% percentile
 
numerical_feature_cols = [
    c for c in df.select_dtypes(include=[np.number]).columns
    if c not in ([TARGET_COL] + SECONDARY_TARGETS + flag_cols + ["month"])
]
 
clip_report = []
for col in numerical_feature_cols:
    Q1  = df[col].quantile(0.01)
    Q99 = df[col].quantile(0.99)
    n_clipped = ((df[col] < Q1) | (df[col] > Q99)).sum()
    if n_clipped > 0:
        df[col] = df[col].clip(lower=Q1, upper=Q99)
        clip_report.append(f"  {col}: {n_clipped} nilai di-clip ke [{Q1:.4f}, {Q99:.4f}]")
 
print(f"Kolom yang mengalami clipping (1%-99% percentile): {len(clip_report)}")
for line in clip_report[:15]:
    print(line)
if len(clip_report) > 15:
    print(f"  ... dan {len(clip_report) - 15} kolom lainnya")
 
print(f"\nTarget '{TARGET_COL}' stats (tidak di-clip):")
print(df[TARGET_COL].describe())
report_lines.append(f"Kolom di-clip outlier: {len(clip_report)}")


STEP 4: DETEKSI & CLIP OUTLIER EKSTREM
Kolom yang mengalami clipping (1%-99% percentile): 99
  ihk_kel_01_mtm: 8 nilai di-clip ke [-10.6350, 3.3341]
  ihk_kel_03_mtm: 8 nilai di-clip ke [-6.6805, 8.2758]
  ihk_kel_06_mtm: 8 nilai di-clip ke [-7.7242, 8.6516]
  gt_harga_beras: 3 nilai di-clip ke [23.0000, 100.0000]
  gt_harga_telur: 3 nilai di-clip ke [26.0000, 100.0000]
  gt_harga_bawang: 3 nilai di-clip ke [20.0000, 100.0000]
  gt_tiket_pesawat: 7 nilai di-clip ke [28.0000, 88.9300]
  gt_tiket_kereta: 7 nilai di-clip ke [8.6900, 98.0000]
  gt_harga_bensin: 4 nilai di-clip ke [18.3800, 100.0000]
  gt_harga_beras_lag1: 3 nilai di-clip ke [23.0000, 100.0000]
  gt_harga_beras_lag2: 3 nilai di-clip ke [23.0000, 100.0000]
  gt_harga_beras_lag3: 3 nilai di-clip ke [23.0000, 100.0000]
  gt_harga_telur_lag1: 3 nilai di-clip ke [26.0000, 100.0000]
  gt_harga_telur_lag2: 3 nilai di-clip ke [26.0000, 100.0000]
  gt_harga_telur_lag3: 3 nilai di-clip ke [26.0000, 100.0000]
  ... dan 84 kolom lainn

In [7]:
# ─── STEP 5: ONE-HOT ENCODING KOTA ────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 5: ONE-HOT ENCODING KOTA")
print("=" * 60)
 
# drop_first=True → hindari multicollinearity (dummy variable trap)
# Reference category: bandung (alphabetically first)
kota_dummies = pd.get_dummies(df["kota"], prefix="kota", drop_first=True, dtype=int)
print("OHE kota columns:", kota_dummies.columns.tolist())
print(kota_dummies.value_counts().reset_index().to_string())
 
df = pd.concat([df, kota_dummies], axis=1)
ohe_kota_cols = kota_dummies.columns.tolist()
report_lines.append(f"OHE kota: {ohe_kota_cols} (reference: bandung)")


STEP 5: ONE-HOT ENCODING KOTA
OHE kota columns: ['kota_jakarta', 'kota_makassar', 'kota_medan', 'kota_surabaya']
   kota_jakarta  kota_makassar  kota_medan  kota_surabaya  count
0             0              0           0              0     74
1             0              0           0              1     74
2             0              0           1              0     74
3             0              1           0              0     74
4             1              0           0              0     74


In [8]:
# ─── STEP 6: VERIFIKASI ANTI-LEAKAGE ──────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 6: VERIFIKASI ANTI-LEAKAGE")
print("=" * 60)
 
# GT tanpa lag (nilai current month) = VALID untuk nowcasting karena GT real-time
# Semua LASI/IHK tanpa lag = target sekunder, bukan fitur
gt_no_lag = [
    c for c in df.columns
    if c.startswith("gt_") and "_lag" not in c
    and "_diff1" not in c and "_pct_change" not in c
]
print(f"Google Trends current month (no lag): {len(gt_no_lag)} kolom")
print("  → VALID untuk nowcasting (tersedia real-time sebelum data BPS)")
print(f"  Contoh: {gt_no_lag[:4]}")
 
# Pastikan target tidak bocor ke fitur
target_in_features = [c for c in df.columns if TARGET_COL in c and c != TARGET_COL]
print(f"\nKolom yang mengandung nama target: {target_in_features}")
print("  → Kolom lag target (ihk_umum_mtm_lag1/2/3) VALID — nilai masa lalu, bukan masa depan")


STEP 6: VERIFIKASI ANTI-LEAKAGE
Google Trends current month (no lag): 9 kolom
  → VALID untuk nowcasting (tersedia real-time sebelum data BPS)
  Contoh: ['gt_harga_beras', 'gt_harga_cabai', 'gt_harga_telur', 'gt_harga_daging_ayam']

Kolom yang mengandung nama target: ['ihk_umum_mtm_lag1', 'ihk_umum_mtm_lag2', 'ihk_umum_mtm_lag3', 'ihk_umum_mtm_roll_mean_3', 'ihk_umum_mtm_roll_std_3', 'ihk_umum_mtm_roll_mean_6', 'ihk_umum_mtm_roll_std_6']
  → Kolom lag target (ihk_umum_mtm_lag1/2/3) VALID — nilai masa lalu, bukan masa depan


In [9]:
# ─── STEP 7: EXPORT ───────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 7: EXPORT FEATURE MATRIX")
print("=" * 60)
 
drop_from_X = ["date", "kota"] + [TARGET_COL] + SECONDARY_TARGETS
feature_cols = [c for c in df.columns if c not in drop_from_X]
print(f"Jumlah fitur X total: {len(feature_cols)}")
 
# Dataset final
df_model = df[["date", "kota"] + feature_cols + [TARGET_COL] + SECONDARY_TARGETS].copy()
 
# Export semua kota
out_all = OUTPUT_DIR / "df_clean.csv"
df_model.to_csv(out_all, index=False)
print(f"\nDisimpan: {out_all}  (shape: {df_model.shape})")
 
# Export per kota (untuk SARIMA)
city_dir = OUTPUT_DIR / "per_kota"
city_dir.mkdir(exist_ok=True)
for kota in sorted(df_model["kota"].unique()):
    df_kota = df_model[df_model["kota"] == kota].reset_index(drop=True)
    out_kota = city_dir / f"df_clean_{kota}.csv"
    df_kota.to_csv(out_kota, index=False)
    print(f"  {out_kota}  (shape: {df_kota.shape})")


STEP 7: EXPORT FEATURE MATRIX
Jumlah fitur X total: 178

Disimpan: D:\Lomba\DATATHON DICODING\inflasiwatch\data\final\output_clean\df_clean.csv  (shape: (370, 185))
  D:\Lomba\DATATHON DICODING\inflasiwatch\data\final\output_clean\per_kota\df_clean_bandung.csv  (shape: (74, 185))
  D:\Lomba\DATATHON DICODING\inflasiwatch\data\final\output_clean\per_kota\df_clean_jakarta.csv  (shape: (74, 185))
  D:\Lomba\DATATHON DICODING\inflasiwatch\data\final\output_clean\per_kota\df_clean_makassar.csv  (shape: (74, 185))
  D:\Lomba\DATATHON DICODING\inflasiwatch\data\final\output_clean\per_kota\df_clean_medan.csv  (shape: (74, 185))
  D:\Lomba\DATATHON DICODING\inflasiwatch\data\final\output_clean\per_kota\df_clean_surabaya.csv  (shape: (74, 185))
